# IMEN266 · HW 3 workspace (Ch.5)

Statement: `hw3.pdf` (PLMS / repo). This notebook is your **computational
workspace** for Parts B–C and your **prompt log** for Part D.
Part A is pen-and-paper first — no cells here on purpose.

▶ Colab: `https://colab.research.google.com/github/youngmko/imen266-2026/blob/main/ch05/homework/HW3.ipynb`

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.linalg import expm
try:
    from imen266.ctmc import CTMC, machine_repair, erlang_loss
except ImportError:                                  # Colab: fetch the course package
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "git+https://github.com/youngmko/imen266-2026.git"], check=True)
    from imen266.ctmc import CTMC, machine_repair, erlang_loss
rng = np.random.default_rng()      # your results should NOT depend on a seed
plt.rcParams.update({"figure.figsize": (7, 4), "axes.grid": True, "grid.alpha": .3})

---
## Part B2 — Correct numbers (formulas on paper, numbers here)

Mean up time 4 days ($\mu=1/4$), mean repair time 1 day ($\lambda=1$), $r=500$,
$c=300$ per day. Fill the limiting distributions by **arc balance** (weights
relative to $p_0$), then the rewards; the generator route is only a check.

In [ ]:
# --- B2 skeleton -------------------------------------------------------------
mu, lam, r, c = 1/4, 1.0, 500.0, 300.0
w1 = np.array([1.0, np.nan, np.nan])        # TODO: case 1 (one repair person):  p1/p0, p2/p0 from lam p0 = mu p1, lam p1 = 2 mu p2
w2 = np.array([1.0, np.nan, np.nan])        # TODO: case 2 (two repair persons): 2 lam p0 = mu p1, lam p1 = 2 mu p2
rew1_rates = np.array([-c, r - c, 2*r])     # reward per day in states 0, 1, 2 (one person)
rew2_rates = np.array([np.nan, np.nan, np.nan])   # TODO: two persons (both charge while working)

if np.isnan(w1).any() or np.isnan(w2).any() or np.isnan(rew2_rates).any():
    print("fill w1, w2 and rew2_rates first")
else:
    p1, p2 = w1 / w1.sum(), w2 / w2.sum()
    print("p (one person):", p1.round(4), "  p (two persons):", p2.round(4))
    print(f"reward one: {p1 @ rew1_rates:.2f}   reward two: {p2 @ rew2_rates:.2f}   rule: r/mu = {r/mu:.0f} vs c/lambda = {c/lam:.0f}")
    # generator check (states = number of machines UP; machine_repair(n, crews, fail_rate, repair_rate))
    print("check one:", machine_repair(2, 1, mu, lam).stationary().round(4), " two:", machine_repair(2, 2, mu, lam).stationary().round(4))

# transient: P(X(1) = 2 | X(0) = 0) with one repair person, and the 3-day failure probability
Q1 = machine_repair(2, 1, mu, lam).Q
print("P(X(1)=2 | X(0)=0) =", expm(Q1 * 1.0)[0, 2].round(4), "   (I+Q)_02 =", (np.eye(3) + Q1)[0, 2])
print("P(fail within 3 days) =", round(1 - math.exp(-mu * 3), 4), "   mu*3 =", mu * 3)

---
## Part B3 — Simulate with competing clocks (no generator)

Each up machine carries an $\exp(\mu)$ failure clock; each machine under repair
carries an $\exp(\lambda)$ repair clock; a machine waiting for the single repair
person carries **no** clock. Advance to the earliest clock, update, repeat.

In [ ]:
# --- B3 skeleton -------------------------------------------------------------
def simulate_clocks(n_crews, T, rng, mu=mu, lam=lam):
    """Two machines; returns the time fractions of 0, 1, 2 machines up over [0, T]."""
    t = 0.0
    up = np.array([True, True])
    fail = rng.exponential(1/mu, 2)                      # failure clocks (inf when down)
    done = np.full(2, np.inf)                            # repair-completion clocks (inf when not in repair)
    waiting = []                                         # machines waiting for a free repair person (FCFS)
    free = n_crews
    occ = np.zeros(3)
    while t < T:
        t_next = min(fail.min(), done.min(), T)
        occ[int(up.sum())] += t_next - t
        t = t_next
        if t >= T:
            break
        if fail.min() <= done.min():                     # a failure
            i = int(fail.argmin()); up[i] = False; fail[i] = np.inf
            if free > 0:
                free -= 1; done[i] = t + rng.exponential(1/lam)
            else:
                waiting.append(i)
        else:                                            # a repair completion
            i = int(done.argmin()); done[i] = np.inf; up[i] = True; fail[i] = t + rng.exponential(1/mu)
            if waiting:
                j = waiting.pop(0); done[j] = t + rng.exponential(1/lam)
            else:
                free += 1
    return occ / occ.sum()

reps, T = 20, 5000
for k, rates in [(1, np.array([-c, r - c, 2*r])), (2, np.array([-2*c, r - c, 2*r]))]:
    est = np.array([simulate_clocks(k, T, rng) @ rates for _ in range(reps)])
    half = 1.96 * est.std(ddof=1) / math.sqrt(reps)
    print(f"{k} repair person(s): reward = {est.mean():.2f} +/- {half:.2f}   (95% CI over {reps} replications)")

---
## Part C — Break the exponential assumption

The simulator below is the one from B3 with the two distributions made
pluggable: `up_sampler(rng)` and `repair_sampler(rng)` return one duration each.
It handles any number of machines and repair persons (FCFS).

**C1 (before running!):** predictions (a)–(c) with one-line reasons: ___

In [ ]:
# --- event-driven simulator for Part C (provided) -----------------------------------
def simulate_repair(n_machines, n_crews, T, up_sampler, repair_sampler, rng):
    """Time fractions of the number of UP machines (0..n) over [0, T]; FCFS repair."""
    n = n_machines; t = 0.0
    up = np.ones(n, bool)
    fail = np.array([up_sampler(rng) for _ in range(n)]); done = np.full(n, np.inf)
    waiting = []; free = n_crews; occ = np.zeros(n + 1)
    while t < T:
        t_next = min(fail.min(), done.min(), T)
        occ[int(up.sum())] += t_next - t
        t = t_next
        if t >= T:
            break
        if fail.min() <= done.min():
            i = int(fail.argmin()); up[i] = False; fail[i] = np.inf
            if free > 0:
                free -= 1; done[i] = t + repair_sampler(rng)
            else:
                waiting.append(i)
        else:
            i = int(done.argmin()); done[i] = np.inf; up[i] = True; fail[i] = t + up_sampler(rng)
            if waiting:
                j = waiting.pop(0); done[j] = t + repair_sampler(rng)
            else:
                free += 1
    return occ / occ.sum()

m_up, m_rep = 2.0, 1.0                                   # mean up time, mean repair time (days)
samplers = {
    "exponential":   lambda mean: (lambda r: r.exponential(mean)),
    "Erlang(4)":     lambda mean: (lambda r: r.gamma(4, mean/4)),
    "deterministic": lambda mean: (lambda r: mean),
    "lognormal(cv=1.5)": lambda mean: (lambda r: r.lognormal(math.log(mean) - 0.5*math.log(1 + 1.5**2), math.sqrt(math.log(1 + 1.5**2)))),
}
print("samplers ready; the CTMC reference for (n machines, k crews):")
for n, k in [(1, 1), (2, 1), (2, 2)]:
    print(f"  {n} machine(s), {k} crew(s): p =", machine_repair(n, k, 1/m_up, 1/m_rep).stationary().round(4))

In [ ]:
# --- C2 skeleton: fill the table ------------------------------------------------------
T = 100_000
cases = [(1, 1), (2, 1), (2, 2)]
print(f"{'machines/crews':>15} {'up dist':>18} {'repair dist':>18}   time fractions of # up")
for n, k in cases:
    for up_name, rep_name in [("exponential", "exponential"), ("exponential", "deterministic"),
                              ("exponential", "Erlang(4)"), ("deterministic", "exponential"),
                              ("lognormal(cv=1.5)", "exponential")]:
        occ = simulate_repair(n, k, T, samplers[up_name](m_up), samplers[rep_name](m_rep), rng)
        print(f"{n}/{k:>13} {up_name:>18} {rep_name:>18}   {occ.round(3)}")
    print(f"{'':>15} {'CTMC':>18} {'':>18}   {machine_repair(n, k, 1/m_up, 1/m_rep).stationary().round(3)}\n")
# TODO: mark which rows agree with the CTMC (means only) and which do not; relate to your C1 predictions.

**C2 write-up (≤5 sentences):** *(double-click to edit)*

> ...

---
## Part C3 (optional) — Is the Erlang loss formula insensitive? Is $M/G/1$?

In [ ]:
# --- C3 skeleton -------------------------------------------------------------
def simulate_loss(K, lam, dur_sampler, n_arrivals, rng):
    """K-channel loss system with Poisson(lam) arrivals and general call durations: fraction of calls lost."""
    t = 0.0; ends = np.full(K, -np.inf); lost = 0
    for _ in range(n_arrivals):
        t += rng.exponential(1/lam)
        free = np.flatnonzero(ends <= t)
        if len(free) == 0:
            lost += 1
        else:
            ends[free[0]] = t + dur_sampler(rng)
    return lost / n_arrivals

K, lam, mean_dur = 4, 3.0, 1.0
print("Erlang B(K=4, a=3) =", round(erlang_loss(lam*mean_dur, K), 4))
for name in ["exponential", "deterministic", "lognormal(cv=1.5)"]:
    print(f"  {name:>18}: lost fraction = {simulate_loss(K, lam, samplers[name](mean_dur), 200_000, rng):.4f}")

# TODO (M/G/1): single server, infinite queue, rho = lam * mean_dur < 1. Simulate the mean number in system for the three
#      duration distributions and compare with the Pollaczek-Khinchine formula L = rho + rho^2 (1 + cv^2) / (2 (1 - rho)).

---
## Part D — Prompt log (keep it whenever AI was used)

| # | Where I used AI | Prompt (verbatim or faithful summary) | What it returned | What I verified / corrected |
|---|---|---|---|---|
| 1 |  |  |  |  |
| 2 |  |  |  |  |
| 3 |  |  |  |  |

**Reflection (2–3 sentences):** where was the AI most useful, and where was it
least trustworthy, on this homework?

> ...